The below code is largely copied from this tutorial from https://jonathansoma.com/words/olmocr-on-macos-with-lm-studio.html with slight edits. Before running the code chunks below be sure to start a local server with the olmocr-7b-0225-preview model loaded. This is the only version I have confirmed it with.

In [ ]:
from openai import OpenAI
# initiate server
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio", timeout=60)

In [ ]:
from pprint import pprint
from olmocr.pipeline import build_page_query

## construct query 
query = await build_page_query("sample.pdf", page=1,target_longest_image_dim=1024)

query

In [ ]:
query['model'] = 'allenai_olmocr-7b-1025'
response = client.chat.completions.create(**query)

import re

def strip_front_matter(text: str) -> str:
    # remove the first front-matter block delimited by --- lines
    return re.sub(r"^---\s*\n.*?\n---\s*\n", "", text, flags=re.DOTALL)

clean = strip_front_matter(response.choices[0].message.content)
print(clean)

In [18]:
import json
import yaml
from pathlib import Path
from tqdm.notebook import trange

from pypdf import PdfReader
from olmocr.pipeline import build_page_query

async def process_page(filename, page_num):
    """Process a single PDF page and extract its transcription as JSON"""
    query = await build_page_query(filename,
                                   page=page_num,
                                   target_longest_image_dim=1024)
    query['model'] = 'allenai_olmocr-7b-1025'
    response = client.chat.completions.create(**query)
    clean = strip_front_matter(response.choices[0].message.content)
    print(f"Page {page_num} - Content length: {len(clean) if clean else 0}")
    
    # Return JSON object with filename, page_num, text (clean), and token counts
    return {
        "filename": filename,
        "page_num": page_num,
        "text": clean,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }

# Process single PDF file
filename = "sample.pdf"
reader = PdfReader(filename)
num_pages = reader.get_num_pages()
results = []

for page_num in trange(1, num_pages + 1):
    result = await process_page(filename, page_num)
    results.append(result)

  0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 1 - Content length: 313


In [20]:
print(results)

[{'filename': 'sample.pdf', 'page_num': 1, 'text': "Name: John Arquette\n\nIndian name: Puyallup\n\nTribe: Puyallup\n\nAge: 19\n\nBlood: \\( \\frac{1}{2} \\)\n\nAgency:\n\nFather: I. Isaac Arquette\n\nArrived: 7-7-'96\n\nDeparted: 7-8-'81\n\nCause: Conduct.\n\nClass entered:\n\nClass left:\n\nTrade:\n\nOuting:\n\nCharacter:\n\nMarried:\n\nDeceased:\n\nRemarks:\nYAWMAN & ERBE MFG. CO., ROCHESTER, N.Y.", 'input_tokens': 945, 'output_tokens': 146}]


In [ ]:
async def process_and_save_directory(input_dir, output_file="test_batch_outputs/output.jsonl"):
    """
    Process all PDF files in a directory and save results to a JSONL file.
    
    Args:
        input_dir: Directory containing PDFs to process
        output_file: Path to output JSONL file (one JSON object per line)
    
    Returns:
        List of result dicts
    """
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith('.pdf')]
    
    if not pdf_files:
        print(f"No PDF files found in {input_dir}")
        return []
    
    all_results = []
    output_path = Path(output_file)
    output_path.parent.mkdir(exist_ok=True, parents=True)
    
    with open(output_path, 'w') as outf:
        for pdf_file in pdf_files:
            pdf_path = os.path.join(input_dir, pdf_file)
            print(f"\nProcessing: {pdf_file}")
            
            reader = PdfReader(pdf_path)
            num_pages = reader.get_num_pages()
            
            for page_num in trange(1, num_pages + 1):
                result = await process_page(pdf_path, page_num)
                all_results.append(result)
                # Write each result as a line in JSONL file
                outf.write(json.dumps(result) + '\n')
    
    print(f"\nProcessed {len(all_results)} pages from {len(pdf_files)} files")
    print(f"Results saved to {output_path}")
    
    return all_results

In [ ]:
process_and_save_directory("test_batch", )


Processing: NARA_1327_b137_f5385_0.pdf


  0%|          | 0/4 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 1 - Raw content length: 540
  in "<unicode string>", line 2, column 1:
    primary_language: en
    ^
but found another document
  in "<unicode string>", line 7, column 1:
    ---
    ^


INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 1 - Raw content length: 864
  in "<unicode string>", line 2, column 1:
    primary_language: en
    ^
but found another document
  in "<unicode string>", line 7, column 1:
    ---
    ^


INFO:httpx:HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Page 2 - Raw content length: 866
  in "<unicode string>", line 2, column 1:
    primary_language: en
    ^
but found another document
  in "<unicode string>", line 7, column 1:
    ---
    ^


CancelledError: 

In [ ]:
from IPython.display import display, HTML
import webbrowser

def create_comparison_html(pdf_file, transcription, page_num=1):
    """Create HTML for side-by-side comparison of PDF and transcription"""
    html = f"""
    <div style="display: flex; gap: 20px; margin: 20px 0;">
        <div style="flex: 1; border: 1px solid #ccc; padding: 10px;">
            <h3>Original PDF: {pdf_file} (Page {page_num})</h3>
            <iframe src="{pdf_file}" width="100%" height="600px"></iframe>
        </div>
        <div style="flex: 1; border: 1px solid #ccc; padding: 10px; overflow-y: auto; max-height: 600px;">
            <h3>Transcription</h3>
            <p style="white-space: pre-wrap; font-family: monospace; font-size: 12px;">
                {transcription}
            </p>
        </div>
    </div>
    """
    return html

def display_comparison(pdf_file, transcription, page_num=1):
    """Display side-by-side comparison in notebook"""
    html = create_comparison_html(pdf_file, transcription, page_num)
    display(HTML(html))

# Example: Display first PDF's transcription
if batch_results:
    first_pdf = list(batch_results.keys())[0]
    first_results = batch_results[first_pdf]
    if first_results:
        transcription = get_transcription(first_results[0])
        display_comparison(os.path.join(PDF_INPUT_DIR, first_pdf), transcription)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Initialize evaluation storage
evaluations = {}

def create_evaluation_form(pdf_file, page_num, transcription_preview):
    """Create an interactive evaluation form"""
    
    # Header
    title = widgets.HTML(f"<h3>Evaluate: {pdf_file} - Page {page_num}</h3>")
    preview = widgets.HTML(f"<p><strong>Transcription Preview:</strong><br><code>{transcription_preview[:200]}...</code></p>")
    
    # Evaluation options
    quality_dropdown = widgets.Dropdown(
        options=[('Select...', None), ('Excellent', 5), ('Good', 4), ('Acceptable', 3), ('Poor', 2), ('Unusable', 1)],
        value=None,
        description='Quality:',
        style={'description_width': '100px'}
    )
    
    comments_text = widgets.Textarea(
        placeholder='Add any comments about the transcription...',
        description='Comments:',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='100%', height='100px')
    )
    
    # Buttons
    submit_btn = widgets.Button(description='Submit Evaluation', button_style='success')
    skip_btn = widgets.Button(description='Skip', button_style='warning')
    
    def on_submit_click(b):
        if quality_dropdown.value is None:
            print("Please select a quality rating")
            return
        
        eval_key = f"{pdf_file}_page_{page_num}"
        evaluations[eval_key] = {
            'pdf_file': pdf_file,
            'page_num': page_num,
            'quality': quality_dropdown.value,
            'comments': comments_text.value,
            'timestamp': datetime.now().isoformat()
        }
        
        save_evaluations_to_csv()
        print(f"✓ Evaluation saved for {pdf_file} - Page {page_num}")
        # Clear form
        quality_dropdown.value = None
        comments_text.value = ''
    
    def on_skip_click(b):
        print(f"Skipped {pdf_file} - Page {page_num}")
    
    submit_btn.on_click(on_submit_click)
    skip_btn.on_click(on_skip_click)
    
    # Layout
    button_box = widgets.HBox([submit_btn, skip_btn])
    form = widgets.VBox([title, preview, quality_dropdown, comments_text, button_box])
    
    return form

def save_evaluations_to_csv():
    """Save evaluations to CSV file"""
    os.makedirs(os.path.dirname(EVAL_OUTPUT_FILE), exist_ok=True)
    
    with open(EVAL_OUTPUT_FILE, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['pdf_file', 'page_num', 'quality', 'comments', 'timestamp'])
        writer.writeheader()
        for eval_data in evaluations.values():
            writer.writerow(eval_data)
    
    print(f"Evaluations saved to {EVAL_OUTPUT_FILE}")

def review_next_item():
    """Interactive review function - display each PDF page with evaluation form"""
    pdf_files = list(batch_results.keys())
    
    for pdf_file in pdf_files:
        page_results = batch_results[pdf_file]
        
        for page_num, result in enumerate(page_results, start=1):
            transcription = get_transcription(result)
            
            # Display comparison
            display_comparison(os.path.join(PDF_INPUT_DIR, pdf_file), transcription, page_num)
            
            # Display evaluation form
            form = create_evaluation_form(pdf_file, page_num, transcription)
            display(form)
            
            print("-" * 60)

In [ ]:
# Start the interactive review process
print("Starting interactive evaluation...")
print(f"Found {len(batch_results)} PDF files to review\n")
review_next_item()